# AAR Legibility PVG — experiments (v2)

Working notebook. **All code now lives in the repo** — nothing is written from
this notebook any more. Edit files in the repo, commit, push, and re-run
section 1 (or `git pull` in place).

**Before anything: Runtime → Change runtime type → T4 GPU.** Then run order: **1 → 2 → 3 → 4**, then 5 for results.
Rough T4 timings: setup 2 min, data 5 s, gate + baselines 3 min, smoke test 5-10 min, full 10×3 run 3-4 h (resumable).

1. Setup — clone the repo, install deps, check the GPU
2. Data — build the ≥100-pair dataset with the 20% held-out split
3. Gate + baselines — the verifier spot check that must pass before any training
4. Runs — smoke test, then 10 rounds × 3 seeds with per-round checkpoints
5. Results — per-round table with the diagnostics, and the curves

Why v2 exists (Sept 2026): the verifier was completing the raw prompt instead
of answering it (both spot-check cases returned the echoed answer menu), the
substring parser then accepted everything, and rounds 1–4 were noise.
Verdicts are now log-prob comparisons under the chat template, the spot check
is a hard gate, every round logs unparseable share / held-out accept rate /
honest-minus-sneaky reward gap and aborts on its own when those go bad.


## 1. Setup

Clones fresh every session. `PROJECT_ROOT` is found rather than hardcoded:
the directory name has a space in it and has moved before.

Until the PR into `gunasti2002/Automated_R_leg_NoAPI` is merged, this clones
the fork branch that carries the fix. After the merge, switch `REPO_URL` to
Brandon's repo and `BRANCH` to `main`.


In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/varchanaiyer/Automated_R_leg_NoAPI.git"   # after merge: https://github.com/gunasti2002/Automated_R_leg_NoAPI.git
BRANCH    = "fix/verifier-scoring-and-gates"                               # after merge: main
CLONE_DIR = Path("/content/Automated_R_leg_NoAPI")

%cd /content
!rm -rf {CLONE_DIR}
!git clone --branch {BRANCH} {REPO_URL} {CLONE_DIR}

matches = sorted(CLONE_DIR.rglob("config.py"))
assert matches, f"No config.py found under {CLONE_DIR}"
PROJECT_ROOT = matches[0].parent
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
!ls


In [ ]:
# torch + matplotlib ship with Colab; do not reinstall torch (CUDA wheel mismatch risk).
!pip install -q jinja2 "transformers>=4.50" peft accelerate

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


## 2. Data

`data/source_findings/` holds the 5 real findings (Wen, Qiu, et al.,
*Automated Weak-to-Strong Researcher*, Anthropic Alignment Science Blog,
2026) — they are committed now, no need to rewrite them here.

The builder adds synthetic weak-to-strong records up to `--target-pairs`,
writes one honest and one sneaky row per record, records the perturbation
that produced each sneaky row, and splits 20% of RECORDS out as held-out
with a fixed seed. Roughly 40% of the sneaky rows are "true numbers,
misleading framing" cases that a number-matching rule check cannot catch.


In [ ]:
!python data/build_findings_dataset.py --target-pairs 120 --seed 42 \
    --out data/findings_pvg_dataset.jsonl --heldout-out data/findings_pvg_heldout.jsonl \
    --manifest-out data/dataset_manifest.json
!cat data/dataset_manifest.json


## 3. Gate + baselines

The verifier spot check. The two fixed cases must get *different* verdicts;
the 20-item hand-checked set (`data/spot_check_set.jsonl`) must be >0.70
accurate with an accept rate in [0.3, 0.7]. `run_one_round.py` refuses to
train (exit 2) if the verifier that plays round 1 fails this. Zero-shot
Qwen2.5-1.5B does NOT clear the 20-item bar (it accepts ~75% of it); the
warmed-up verifier (2 epochs on the labeled train split) is what has to.


In [ ]:
import sys, json, torch
sys.path.insert(0, ".")
from config import PVGConfig
from training import pvg_loop

cfg = PVGConfig()
spot_rows = pvg_loop.load_dataset(cfg.spot_check_set_path)

print("=== untrained verifier ===")
report = pvg_loop.run_spot_check(cfg, spot_set_rows=spot_rows, enforce_set=False)
for c in report["cases"]:
    print(f"  {c['case']}: {c['verdict']}  P(sound)={c['p_sound']:.3f}")
print("  20-item set:", {k: report[k] for k in ("set_accuracy", "set_accept_rate", "set_auroc")})


In [ ]:
# Rule-check and zero-shot-Qwen baselines on the held-out split (+ the 20-item set).
!python eval/baselines.py --out eval/baselines_heldout.json


In [ ]:
# Drop the notebook's verifier so the training subprocesses get the full card.
import gc, sys, torch
if "training.pvg_loop" in sys.modules:
    sys.modules["training.pvg_loop"]._VERIFIER_CACHE.clear()
gc.collect(); torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"GPU free: {free/1024**3:.2f} / {total/1024**3:.2f} GiB")
assert free / total > 0.9, "Something is still holding GPU memory — restart the runtime."


## 4. Runs

- `training/pvg_loop.py --rounds 1` — smoke test in one process, no checkpointing.
- `run_experiment.py` — 10 rounds × 3 seeds, one subprocess per round with
  checkpoints under `training/checkpoint/seed_<seed>/`. Re-run the same cell
  to resume after a disconnect. `--fresh` archives any existing checkpoint
  (the pre-fix rounds are archived automatically either way — they are not
  usable).

Exit codes: 2 = gate refused to train, 3 = a seed aborted itself (held-out
accept rate > 0.9 or reward gap ≤ 0). Look at the last row of that seed's
`round_history.jsonl` for the reason.

Run the smoke test first. If it exits 2 the gate refused (verifier not discriminating after warm-up) — stop and report, do not start the full run.


In [ ]:
SEEDS          = [42, 43, 44]
TOTAL_ROUNDS   = 10
CHECKPOINT_DIR = "training/checkpoint"


In [ ]:
# Smoke test — 1 round, no checkpointing. Confirms the wiring works end to end.
!python training/pvg_loop.py --rounds 1 --seed 42
!cat training/pvg_round_history.jsonl


In [ ]:
# Full run. Safe to interrupt; re-run this cell to continue where it stopped.
seeds = " ".join(str(s) for s in SEEDS)
!python run_experiment.py --seeds {seeds} --rounds {TOTAL_ROUNDS} --checkpoint-dir {CHECKPOINT_DIR}


## 5. Results

`run_one_round.py` appends one row per completed round to
`training/checkpoint/seed_<seed>/round_history.jsonl` (and a combined
`training/checkpoint/round_history.jsonl`). Each row carries the classic two
metrics plus the diagnostics: `unparseable_share`, `heldout_accept_rate`,
`heldout_accuracy`, `heldout_auroc`, `reward_gap`, role fidelity, spot-check
accuracy, and `aborted`.


In [ ]:
!python eval/plot_rounds.py --checkpoint-dir {CHECKPOINT_DIR} --out {CHECKPOINT_DIR}/pvg_curves.png
from IPython.display import Image, display
display(Image(filename=f"{CHECKPOINT_DIR}/pvg_curves.png"))


In [ ]:
import json, pandas as pd
from pathlib import Path
rows = []
for f in sorted(Path(CHECKPOINT_DIR).glob("seed_*/round_history.jsonl")):
    rows += [json.loads(l) for l in f.read_text().splitlines() if l.strip()]
df = pd.DataFrame(rows)
cols = ["seed", "round_number", "helpful_prover_accuracy", "verifier_robustness", "unparseable_share",
        "heldout_accept_rate", "heldout_accuracy", "heldout_auroc", "reward_gap",
        "role_fidelity_helpful", "role_fidelity_sneaky", "aborted", "elapsed_s"]
df[[c for c in cols if c in df.columns]]


### Persisting results

Colab's disk is gone when the runtime resets. To keep a run, copy
`training/checkpoint/` to Drive, or commit `round_history.jsonl` +
`pvg_curves.png` and push:

```
!git config user.email you@example.com && git config user.name you
!git add training/checkpoint/*/round_history.jsonl training/checkpoint/pvg_curves.png eval/baselines_heldout.json
!git commit -m "results: 10x3 run" && git push https://<token>@github.com/<you>/Automated_R_leg_NoAPI.git HEAD
```
